## Classification

To run this script, you need the following files found in the /data directory:
- "final_labels_SG1.xlsx"
- "final_labels_SG2.xlsx"

## Imports and set-up

In [ ]:
# Resolve paths relative to the repository root so this notebook runs from the repo
# root, from notebooks/, or on Colab after cloning the repo:
#   !git clone https://github.com/yasmeenhany/media-bias-detection.git
#   %cd media-bias-detection
import os
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
os.chdir(REPO_ROOT)
print("Working directory:", REPO_ROOT)

## Preprocessing

### Basic Preprocessing

In [18]:
import pandas as pd

PATH_sg1 = "data/final_labels_SG1.xlsx"
PATH_sg2 = "data/final_labels_SG2.xlsx"
df_sg1 = pd.read_excel(PATH_sg1)
df_sg2 = pd.read_excel(PATH_sg2)
df_sg1.rename(columns={'text': 'sentence', 'label_bias': 'Label_bias'}, inplace=True)
df_sg2.rename(columns={'text': 'sentence', 'label_bias': 'Label_bias'}, inplace=True)
df_sg1.head()

,sentence,news_link,outlet,topic,type,Label_bias,label_opinion,biased_words
0,The Republican president assumed he was helpin...,http://www.msnbc.com/rachel-maddow-show/auto-i...,msnbc,environment,left,Biased,Expresses writer’s opinion,[]
1,Though the indictment of a woman for her own p...,https://eu.usatoday.com/story/news/nation/2019...,usa-today,abortion,center,Non-biased,Somewhat factual but also opinionated,[]
2,Ingraham began the exchange by noting American...,https://www.breitbart.com/economy/2020/01/12/d...,breitbart,immigration,right,No agreement,No agreement,['flood']
3,The tragedy of America’s 18 years in Afghanist...,http://feedproxy.google.com/~r/breitbart/~3/ER...,breitbart,international-politics-and-world-news,right,Biased,Somewhat factual but also opinionated,"['tragedy', 'stubborn']"
4,The justices threw out a challenge from gun ri...,https://www.huffpost.com/entry/supreme-court-g...,msnbc,gun-control,left,Non-biased,Entirely factual,[]


In [19]:
# binarize classification problem
df_sg1 = df_sg1[df_sg1['Label_bias']!='No agreement']
df_sg1 = df_sg1[df_sg1['Label_bias'].isna()==False]
df_sg1.replace(to_replace='Biased', value=1, inplace=True)
df_sg1.replace(to_replace='Non-biased', value=0, inplace=True)

df_sg2 = df_sg2[df_sg2['Label_bias']!='No agreement']
df_sg2.replace(to_replace='Biased', value=1, inplace=True)
df_sg2.replace(to_replace='Non-biased', value=0, inplace=True)

<ipython-input-19-3610be668419>:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sg1.replace(to_replace='Non-biased', value=0, inplace=True)
<ipython-input-19-3610be668419>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sg2.replace(to_replace='Biased', value=1, inplace=True)
<ipython-input-19-3610be668419>:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

### EDA

In [ ]:
df_sg1.head()

,sentence,news_link,outlet,topic,type,Label_bias,label_opinion,biased_words
0,The Republican president assumed he was helpin...,http://www.msnbc.com/rachel-maddow-show/auto-i...,msnbc,environment,left,1,Expresses writer’s opinion,[]
1,Though the indictment of a woman for her own p...,https://eu.usatoday.com/story/news/nation/2019...,usa-today,abortion,center,0,Somewhat factual but also opinionated,[]
3,The tragedy of America’s 18 years in Afghanist...,http://feedproxy.google.com/~r/breitbart/~3/ER...,breitbart,international-politics-and-world-news,right,1,Somewhat factual but also opinionated,"['tragedy', 'stubborn']"
4,The justices threw out a challenge from gun ri...,https://www.huffpost.com/entry/supreme-court-g...,msnbc,gun-control,left,0,Entirely factual,[]
5,A review of his posts in online message boards...,https://eu.usatoday.com/story/news/nation/2020...,usa-today,white-nationalism,center,1,Entirely factual,['plant']


In [ ]:
# Number of Labels

df_sg1['Label_bias'].value_counts()

,count
Label_bias,
0,800
1,746


In [ ]:
for i, sentence in enumerate(df_sg1[df_sg1["Label_bias"]==1]['sentence']):
    print(f"{i+1}. {sentence}")
    if i == 30: break

1. The Republican president assumed he was helping the industry at the expense of the environment – a trade-off Trump was happy to make since he rejects climate science anyway. 
2. The tragedy of America’s 18 years in Afghanistan has been a stubborn refusal to admit the Afghan government is incapable of standing on its own, no matter how much money is poured into it, how much training its troops are given, or how many of its battles American soldiers fight.
3. A review of his posts in online message boards revealed he wanted to plant neo-Nazi propaganda inside Nevada middle and high schools.
4. Politicians have no business directing or defining patriotism — especially when their rhetoric sounds like 1950s-era Soviet sloganeering.
5. Last week, Ankara broke the 2015 agreement with Brussels in which it would stop the illegal flow of migrants through Turkey in exchange for money and continuing EU accession talks, and opened the gates which resulted in thousands of migrants amassing on the

In [ ]:
for i, sentence in enumerate(df_sg1[df_sg1["Label_bias"]==0]['sentence']):
    print(f"{i+1}. {sentence}")
    if i == 30: break

1. Though the indictment of a woman for her own pregnancy loss is unusual in Alabama, it is not unusual for prosecutors to charge people with murder even if they never killed anyone. 
2. The justices threw out a challenge from gun rights groups.
3. After the airstrike that killed Iranian Gen. Qasem Soleimani, there were reports that U.S. Customs and Border Patrol detained dozens of people at the Blaine, Washington, port of entry over the weekend.
4. Investigators believe parents would use falsified records so their children could attend schools that would otherwise require all students to be vaccinated, Dart told WBBM-TV.
5. Despite repeated setbacks, Smart made history as the first American to reach No. 1 in the world in saber fencing and later was on the silver-medal winning U.S. team in the 2008 Olympics.
6. In the wake of the El Paso shooting, in which 22 were killed and 24 injured, the discovery of a racist manifesto that may be linked to the shooter has pushed white supremacist i

### Data Augmentation

#### T5 Data Augmentation

In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load pre-trained T5 model and tokenizer
model_name = "Vamsi/T5_Paraphrase_Paws"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Function to augment text by paraphrasing
def paraphrase_text(text, temperature=1.0, top_k=50, max_length=200):
    # Prepare input in T5 format with "paraphrase:" prefix
    input_text = f"paraphrase: {text}"

    # Tokenize the input text
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)

    # Generate paraphrase with randomness
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            max_length=max_length,
            do_sample=True,          # Enable sampling
            temperature=temperature, # Sampling temperature
            top_k=top_k,             # Top-k sampling
            early_stopping=True
        )

    # Decode the paraphrased output
    paraphrased_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return paraphrased_text


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Test for suitable temperature

In [ ]:
# Example usage
original_text = "President Donald Trump offered an unusual warning to Virginia farmers on Tuesday, suggesting that gun control will leave their potatoes defenseless."
augmented_text = paraphrase_text(original_text)

temperatures = [0.5, 0.8, 1.0, 1.2, 1.5, 1.8, 2.0]
print(f"Original Text: {original_text}")
for temp in temperatures:
    augmented_text = paraphrase_text(original_text, temperature=temp)
    print(f"Temperature: {temp}, Augmented Text: {augmented_text}")

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


Original Text: President Donald Trump offered an unusual warning to Virginia farmers on Tuesday, suggesting that gun control will leave their potatoes defenseless.
Temperature: 0.5, Augmented Text: On Tuesday, President Donald Trump offered an unusual warning to Virginia farmers, suggesting that gun control will leave their potatoes defenseless .
Temperature: 0.8, Augmented Text: On Tuesday, president Donald Trump offered an unusual warning to Virginia farmers, suggesting that gun control will make their potatoes defenseless .
Temperature: 1.0, Augmented Text: On Tuesday, President Donald Trump offered an unusual warning to Virginia farmers, suggesting that gun control will leave their potatoes defenseless .
Temperature: 1.2, Augmented Text: On Tuesday, president Trump offered an unusual warning to Virginia farmers, suggesting that gun control will leave their potatoes defenseless .
Temperature: 1.5, Augmented Text: On Tuesday, President Trump offered an unusual warning to the Virginia

In [ ]:
# Example usage
original_text = "As the world’s scientists and pharmaceutical companies seek a cure for the coronavirus, other anti-vaxxers are gearing up for a fight against any potential new vaccine."
augmented_text = paraphrase_text(original_text)

temperatures = [0.5, 0.8, 1.0, 1.2, 1.5, 1.8, 2.0]
print(f"Original Text: {original_text}")
for temp in temperatures:
    augmented_text = paraphrase_text(original_text, temperature=temp)
    print(f"Temperature: {temp}, Augmented Text: {augmented_text}")

Original Text: As the world’s scientists and pharmaceutical companies seek a cure for the coronavirus, other anti-vaxxers are gearing up for a fight against any potential new vaccine.
Temperature: 0.5, Augmented Text: As the world’s scientists and pharmaceutical companies seek a cure for the coronavirus, other anti-vaxxers are gearing up for a fight against any potential new vaccine .
Temperature: 0.8, Augmented Text: As scientists and pharmaceutical companies around the world seek a cure for the coronavirus, other anti-vaxxers are gearing up for a fight against any potential new vaccine .
Temperature: 1.0, Augmented Text: As the world’s scientists and pharmaceutical companies seek a cure for the coronavirus , other anti-vaxxers are gearing up for a fight against any potential new vaccine .
Temperature: 1.2, Augmented Text: While the world’s scientists and pharmaceutical companies seek a cure for coronavirus, other anti-vaxxers are gearing up to fight up against any potential new vacci

Add paraphrased sentences

In [ ]:
from collections import defaultdict
from tqdm import tqdm

augmented_sentences_temp13 = defaultdict(list)

for idx, row in tqdm(list(df_sg1.iterrows())):

    augmented_text = paraphrase_text(row['sentence'], temperature=1.3)
    augmented_sentences_temp13[idx].append(augmented_text)

  0%|          | 0/1546 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
100%|██████████| 1546/1546 [1:26:11<00:00,  3.35s/it]


In [ ]:
with open('data/augmentation/augmented_sentences_temp13.pickle', 'wb') as file:
    pickle.dump(augmented_sentences_temp13, file, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import pickle

with open('data/augmentation/augmented_sentences_temp15.pickle', 'rb') as file:
    augmented_sentences_temp15 = pickle.load(file)

In [ ]:
cnt = 0
for key, item in augmented_sentences_temp15.items():
    print(key, item)
    if key == 2: break

print(cnt)

In [ ]:
from collections import defaultdict

idx2key = defaultdict(int)

for idx, key in enumerate(sorted(augmented_sentences_temp15.keys())):
    idx2key[idx] = key

#### ChatGPT Generated Sentences

Sentences were created with a prompt, and then were saved as a pickle file.

Prompt Version 1.

I need you to make me politically biased sentences. Make me 100 of them. Format it in a python list

In [1]:
import pickle

In [3]:
with open("data/augmentation/prompt1_biased_sentences.pickle", "rb") as f:
    v1_biased_sentences = pickle.load(f)

print(len(v1_biased_sentences))
print(v1_biased_sentences[0])

100
The Democratic senator continues to push her socialist agenda, disregarding the economic consequences of such policies.


In [16]:
with open("data/augmentation/prompt1_unbiased_sentences.pickle", "rb") as f:
    v1_unbiased_sentences = pickle.load(f)

print(len(v1_unbiased_sentences))
print(v1_unbiased_sentences[0])

100
Though the indictment of a woman for her own pregnancy loss is unusual in Alabama, it is not unusual for prosecutors to charge people with murder even if they never killed anyone.


Prompt Version 2.

Make me 100 politically biased sentences.
Format them into a python list.
here are 10 examples.

{10 example sentences}

In [28]:
with open("data/augmentation/prompt2_biased_sentences.pickle", "rb") as f:
    v2_biased_sentences = pickle.load(f)

print(len(v2_biased_sentences))
print(v2_biased_sentences[0])

100
The Republican president assumed he was helping the industry at the expense of the environment – a trade-off Trump was happy to make since he rejects climate science anyway.


In [30]:
with open("data/augmentation/prompt2_unbiased_sentences.pickle", "rb") as f:
    v2_unbiased_sentences = pickle.load(f)

print(len(v2_unbiased_sentences))
print(v2_unbiased_sentences[0])

100
Though the indictment of a woman for her own pregnancy loss is unusual in Alabama, it is not unusual for prosecutors to charge people with murder even if they never killed anyone.


Prompt Version 3.

Make me 100 politically biased sentences.
Focus on nuanced bias types rather than overt partisan language.

Here are examples for you to refer to.

30 Biased sentences
{randomly chosen biased sentence list}

30 Unbiased sentences
{randomly chosen unbiased sentence list}

List them into a python list

In [22]:
cnt = 1

for i, row in df_sg1.iterrows():

    if row['Label_bias'] == 0:

        print(f"{cnt}. ", row['sentence'])
        cnt += 1

    if cnt == 30: break

1.  Though the indictment of a woman for her own pregnancy loss is unusual in Alabama, it is not unusual for prosecutors to charge people with murder even if they never killed anyone. 
2.  The justices threw out a challenge from gun rights groups.
3.  After the airstrike that killed Iranian Gen. Qasem Soleimani, there were reports that U.S. Customs and Border Patrol detained dozens of people at the Blaine, Washington, port of entry over the weekend.
4.  Investigators believe parents would use falsified records so their children could attend schools that would otherwise require all students to be vaccinated, Dart told WBBM-TV.
5.  Despite repeated setbacks, Smart made history as the first American to reach No. 1 in the world in saber fencing and later was on the silver-medal winning U.S. team in the 2008 Olympics.
6.  In the wake of the El Paso shooting, in which 22 were killed and 24 injured, the discovery of a racist manifesto that may be linked to the shooter has pushed white suprema

In [24]:
with open("data/augmentation/prompt3_biased_sentences.pickle", "rb") as f:
    v3_biased_sentences = pickle.load(f)

print(len(v3_biased_sentences))
print(v3_biased_sentences[0])

100
The Republican president assumed he was helping the industry at the expense of the environment – a trade-off he was happy to make since he rejects climate science anyway.


In [26]:
with open("data/augmentation/prompt3_unbiased_sentences.pickle", "rb") as f:
    v3_unbiased_sentences = pickle.load(f)

print(len(v3_unbiased_sentences))
print(v3_unbiased_sentences[0])

100
The justices threw out a challenge from gun rights groups.


Prompt version 4.

Make me 100 politically biased sentences.
Focus on nuanced bias types rather than overt partisan language.

Here are examples for you to refer to. Generate sentences that have very similar nuances as the biased sentence examples, but change the content of the sentences to various political topics and refrain from using exactly same words. Also, try to make the sentence as distant as possible to the unbiased sentence examples.

30 Biased sentences
{high confidence biased sentence list}

30 Unbiased sentences
{high confidence unbiased sentence list}

List them into a python list

Prompt version 4.

Make me 100 politically unbiased sentences.
Focus on nuanced bias types rather than overt partisan language.

Here are examples for you to refer to. Generate sentences that have very similar nuances as the unbiased sentence examples, but change the content of the sentences to various political topics and refrain from using exactly same words. Also, try to make the sentence as distant as possible to the biased sentence examples.

30 Biased sentences
{high confidence biased sentence list}

30 Unbiased sentences
{high confidence unbiased sentence list}

List them into a python list

In [32]:
with open("data/augmentation/prompt4_biased_sentences.pickle", "rb") as f:
    v4_biased_sentences = pickle.load(f)

print(len(v4_biased_sentences))
print(v4_biased_sentences[0])

100
Biden’s decision to address climate change aggressively reflects the Democrats' strategy to appease their progressive base, even at the expense of alienating moderate voters.


In [34]:
with open("data/augmentation/prompt4_unbiased_sentences.pickle", "rb") as f:
    v4_unbiased_sentences = pickle.load(f)

print(len(v4_unbiased_sentences))
print(v4_unbiased_sentences[0])

100
The city council voted to allocate funds for rebuilding local infrastructure after last year's storm damage.


Prompt version 5.

Make me 100 politically biased sentences.
Focus on nuanced bias types rather than overt partisan language.

Here are examples for you to refer to. Generate sentences that have very similar nuances as the biased sentence examples, but change the content of the sentences to various political topics and refrain from using exactly same words from the examples.  

These sentences do not use explicit words, but rather use nuanced expressions. Try to do the same with your sentences.

30 Biased sentences
{low confidence biased sentence list}

List them into a python list

In [36]:
with open("data/augmentation/prompt5_biased_sentences.pickle", "rb") as f:
    v5_biased_sentences = pickle.load(f)

print(len(v5_biased_sentences))
print(v5_biased_sentences[0])

100
A high school curriculum focusing on renewable energy solutions — an area some claim is riddled with pseudoscience — has gained traction among progressive educators.


In [38]:
with open("data/augmentation/prompt5_unbiased_sentences.pickle", "rb") as f:
    v5_unbiased_sentences = pickle.load(f)

print(len(v5_unbiased_sentences))
print(v5_unbiased_sentences[0])

100
The ongoing debate over renewable energy subsidies highlights concerns about corporate influence on green initiatives.


In [14]:
biased_sentences = v4_biased_sentences + v5_biased_sentences
unbiased_sentences = v4_unbiased_sentences + v5_unbiased_sentences